In [1]:
import pandas as pd

In [6]:
top14_16_17 = pd.read_csv('top14_2016-2017.csv')
top14_17_18 = pd.read_csv('top14_2017-2018.csv')
top14_18_19 = pd.read_csv('top14_2018-2019.csv')
top14_19_20 = pd.read_csv('top14_2019-2020.csv')

top14_merge = pd.concat([top14_16_17, top14_17_18, top14_18_19, top14_19_20], ignore_index=True)

print(top14_merge["domicile"].unique())

<StringArray>
[                'LOU Rugby',         'Castres Olympique',
     'Union Bordeaux-Bègles',      'Stade Français Paris',
           'Stade Rochelais',          'Stade Toulousain',
          'Aviron Bayonnais',           'Section Paloise',
                  'CA Brive',         'FC Grenoble Rugby',
                 'Racing 92', 'Montpellier Hérault Rugby',
                 'RC Toulon',              'ASM Clermont',
             'Oyonnax Rugby',                   'SU Agen',
             'USA Perpignan']
Length: 17, dtype: str


In [9]:
import csv
from datetime import datetime

# CONFIGURATION
FILE_IN = "top14_2018-2019.csv"
FILE_OUT = "top14_2018-2019_pivot.csv"
SAISON = "2018-2019" # Utilisé pour déduire l'année des dates

def parse_rugby_date(date_str, saison_str):
    annee_depart = int(saison_str.split('-')[0])
    mois_map = {
        'janvier': 1, 'février': 2, 'mars': 3, 'avril': 4, 'mai': 5, 'juin': 6,
        'juillet': 7, 'août': 8, 'septembre': 9, 'octobre': 10, 'novembre': 11, 'décembre': 12
    }
    try:
        parts = date_str.lower().replace(',', '').split()
        # On cherche le jour (numérique) et le mois (texte)
        jour = next(int(p) for p in parts if p.isdigit())
        mois_nom = next(m for m in mois_map if m in parts)
        mois = mois_map[mois_nom]
        
        # Rugby : Si mois <= 7 (juillet), on est sur l'année de fin de saison
        annee = annee_depart + 1 if mois <= 7 else annee_depart
        return f"{jour:02d}/{mois:02d}/{annee}"
    except:
        return date_str

def convert_to_tableau_format(input_path, output_path, saison):
    with open(input_path, "r", encoding="utf-8") as f_in:
        reader = csv.DictReader(f_in)
        
        with open(output_path, "w", encoding="utf-8", newline="") as f_out:
            fieldnames = [
                "journee", "date_formatee", "equipe", "role", 
                "adversaire", "score_equipe", "score_adversaire", 
                "classement_avant_match", "bonus", "resultat"
            ]
            writer = csv.DictWriter(f_out, fieldnames=fieldnames)
            writer.writeheader()

            for row in reader:
                date_clean = parse_rugby_date(row["date"], saison)
                s_dom = int(row["score_dom"])
                s_ext = int(row["score_ext"])

                # Logique Victoire / Défaite / Nul
                res_dom = "Victoire" if s_dom > s_ext else "Défaite" if s_dom < s_ext else "Nul"
                res_ext = "Victoire" if s_ext > s_dom else "Défaite" if s_ext < s_dom else "Nul"

                # Ligne DOMICILE
                writer.writerow({
                    "journee": row["journee"],
                    "date_formatee": date_clean,
                    "equipe": row["domicile"],
                    "role": "Domicile",
                    "adversaire": row["exterieur"],
                    "score_equipe": s_dom,
                    "score_adversaire": s_ext,
                    "classement_avant_match": row["classement_dom"].split("e")[0],  # On garde que le numéro du classement
                    "bonus": row["bonus_dom"],
                    "resultat": res_dom
                })

                # Ligne EXTÉRIEUR
                writer.writerow({
                    "journee": row["journee"],
                    "date_formatee": date_clean,
                    "equipe": row["exterieur"],
                    "role": "Extérieur",
                    "adversaire": row["domicile"],
                    "score_equipe": s_ext,
                    "score_adversaire": s_dom,
                    "classement_avant_match": row["classement_ext"].split("e")[0],  # On garde que le numéro du classement
                    "bonus": row["bonus_ext"],
                    "resultat": res_ext
                })

if __name__ == "__main__":
    convert_to_tableau_format(FILE_IN, FILE_OUT, SAISON)
    print(f"Conversion terminée ! Fichier dispo : {FILE_OUT}")

Conversion terminée ! Fichier dispo : top14_2018-2019_pivot.csv
